In [1]:
# pip install langchain_huggingface
# !pip install langchain_community

In [2]:
# type(chunks[0])

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import spacy
# import faiss
from langchain_google_genai import ChatGoogleGenerativeAI

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_1092\3305012302.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


load the document

In [4]:
data = open(r'D:\qsp GenAI\RAG\machine_learning_2000_sentences.txt').read()
# data

text normalization

convert all characters to lower

In [5]:
data = data.lower()

remove extra space

In [6]:
import re
data = re.sub(r'\s{2,}', ' ', data)

remove 'machine learning statement 1:' pattern

In [7]:
data = re.sub(r'\d+\:', '', data)
# data

In [8]:
# data

expand contraction and abbreviation

In [9]:
import contractions

In [10]:
data = contractions.fix(data)

punctutations

In [11]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [12]:
data = re.sub(r'[^0-9a-zA-Z\s]','', data)

correct spellings using textblob

In [13]:
from textblob import TextBlob

In [14]:
# str(TextBlob(data).correct())

In [15]:
# data = re.sub(r'\d+\.','',data)
data = re.sub(r'\d+\.\b','',data)
# data

spacy lemmatization

In [16]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [17]:
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
# updated_tokens
# if we want to check the word belong to which entity, .ent

In [18]:
data = ' '.join(updated_tokens).strip() # strip to remove extra space, we can also remove characters

chunking

In [19]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 20)

In [20]:
# chunks = splitter.split_text(data)
# chunks = list(set(splitter.split_text(data)))
chunks = splitter.create_documents([data])

In [21]:
print(chunks[0])

page_content='machine learn statement   workflow emphasizing unsupervise'


In [22]:
type(chunks[0])

langchain_core.documents.base.Document

In [23]:
print(chunks[0].page_content)

machine learn statement   workflow emphasizing unsupervise


In [48]:
chunks[0].metadata='data.txt'
chunks[2]

Document(metadata={}, page_content='monitor logistic regression document assumption compare')

In [25]:
chunks[0].metadata={'file_name':'data.txt'}
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learn statement   workflow emphasizing unsupervise'),
 Document(metadata={}, page_content='learning improve carefully validate feature scale'),
 Document(metadata={}, page_content='monitor logistic regression document assumption compare'),
 Document(metadata={}, page_content='result meaningful baseline deployment machine'),
 Document(metadata={}, page_content='learn statement   workflow emphasize reinforcement learning'),
 Document(metadata={}, page_content='improve carefully validate linear regression monitor'),
 Document(metadata={}, page_content='dimensionality reduction documenting assumption compare result'),
 Document(metadata={}, page_content='meaningful baseline deployment machine learn'),
 Document(metadata={}, page_content='statement   workflow emphasize classification improve'),
 Document(metadata={}, page_content='carefully validate neural network monitor randomizedsearchcv'),
 Document(metadata={}, page_co

### Chunk Embeddings (Converts chunks to vectors)

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [27]:
# model_name = "sentence-transformers/all-mpnet-base-v2"  by default it uses this model if not mentioned any
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-miniLM-L6-V2"
)
# llm_model = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash", # 3.5 flash 2.0 flash, 2.5-pro, 3.6-flash
#     # api_key=os.environ["GEMINI_API_KEY"]

# )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5138.98it/s]


In [28]:
# response = llm_model.invoke("Explain GenAI?")

In [29]:
# chunks

In [30]:
vector_db = FAISS.from_documents(
    documents = chunks, # where we store all documents
    embedding = embedding_model
)
vector_db

In [31]:
user_query = 'What is Machine Learning'
r_chunks = vector_db.similarity_search(user_query) # returns directly content not distance and index

In [32]:
# r_chunks = set()
for chunk in r_chunks:
    print(chunk.page_content)

machine learn statement   workflow emphasizing gradient boosting
machine learn statement   workflow emphasizing gradient boosting
machine learn statement   workflow emphasizing gradient boosting
machine learn statement   workflow emphasizing gradient boosting


Retrieval

In [33]:
updated_r_chunks = set()
for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
# updated_r_chunks
R_text = '\n'.join(updated_r_chunks) # it accepts iterables

In [34]:
R_text

'machine learn statement   workflow emphasizing gradient boosting'

if we use llms using langchain, we can use pipeline also
- by using langchain we can create ai agents also.
- we use langchain and langraph to create ai agent
- we won't use huggingface, because we will reach limit. in claude also
- we will download llm models.


Structure the output

In [35]:
# create gemini api key

In [36]:
# pip show langchain

In [37]:
# dimension = chunk_embeddings.shape[1]
# dimension

Normalize

In [38]:
# faiss.normalize_L2(chunk_embeddings)

In [39]:
# index_faiss_db = faiss.IndexFlatIP(dimension)
# index_faiss_db.add(chunk_embeddings)

In [40]:
# def r_search(query,k=3):
#     query_embeddings = embedding_model.encode(query).astype('float32')
#     query_embeddings = query_embeddings.reshape(1,-1)
#     faiss.normalize_L2(query_embeddings)
#     print(query_embeddings.shape)
#     distance,index = index_faiss_db.search(query_embeddings,k=k)
#     R_chunks = [chunks[i] for i in index[0]]
#     R_str = ' '.join(R_chunks)
#     for chunk in R_chunks:
#         print(chunk.page_content)
#     return R_str


In [41]:
# def r_search(query,k=3):
#     query_embeddings = embedding_model.encode(query).astype('float32')
#     query_embeddings = query_embeddings.reshape(1,-1)
#     faiss.normalize_L2(query_embeddings)
#     print(query_embeddings.shape)
#     distance,index = index_faiss_db.search(query_embeddings,k=k)
#     R_chunks = [chunks[i] for i in index[0]]
#     R_str = ' '.join(R_chunks)
#     return R_str

# def g_text(r_search):
#         import os
#         import requests
    
#         API_URL = "https://router.huggingface.co/v1/chat/completions"
    
#         headers = {
#             "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
#         }
#         def query(payload):
#             response = requests.post(API_URL, headers=headers, json=payload)
#             return response.json()
#         prompt = f'''
#                     You're an helpful assistant
#                     Assigned Task for you : Structure my output => {r_search}
#                     Note : 
#                     1) Don't add extra contents just structure mentioned output.
#                     2) If there is mistake in output correct or else keep the original output
#                     with structured result.
#             '''
#         response = query({
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": f'{prompt}'  #JSONDecodeError
#                 }
#             ],
#             "model": "deepseek-ai/DeepSeek-R1:novita"
#         })
    
#         return response

# user_prompt = 'Explain Machine Learning ?'
# user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

# r_response = r_search(user_prompt)
# g_response = g_text(r_response)
# print(g_response)

In [42]:
# change to hugging face

In [43]:
def r_search(query, k=2): 
    # query_embedding = embedding_model.encode(query).astype('float32')
    # query_embedding = query_embedding.reshape(1,-1)
    # faiss.normalize_L2(query_embedding)
    # distance, index = index_faiss_db.search(query_embedding,k=k)
    R_chunks = vector_db.similarity_search(query,k=k)
    # print(R_chunks)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_Text = '\n'.join(R_chunks)
    # print(R_Text)
    return R_Text
    # R_chunks = [chunks[i] for i in index[0]]
    # R_str = ' '.join(R_chunks)
    # return R_str


def g_text(r_search):
    import os

    prompt = f'''
            You're an helpful assistant
            Assigned Task for you:
            Structure my output => {r_search}
    '''
    llm_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # 3.5 flash 2.0 flash, 2.5-pro, 3.6-flash
    # api_key=os.environ["GEMINI_API_KEY"]
        )

    response = llm_model.invoke('Explain GenAI?').content
    return response


user_prompt = 'Explain Machine Learning?'
user_prompt = re.sub(r'[^0-9a-z-A-Z\s]',' ', user_prompt)
r_response = r_search(user_prompt)
# g_response = g_text(r_response)
# print(g_response)
# print()
# print(len(r_response))
# print(r_response[:500])

In [44]:
def r_search(query, k=2): 
    R_chunks = vector_db.similarity_search(query,k=k)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_Text = '\n'.join(R_chunks)
    return R_Text



def g_text(r_search, query):
    import os

    prompt = f'''
            You're an helpful assistant
            Assigned Task for you:
            Structure my output => {r_search} for this input => {query}
            Output structure:
            Input: {query}
            Output: structured output
    '''
    llm_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # 3.5 flash 2.0 flash, 2.5-pro, 3.6-flash
        )

    response = llm_model.invoke(prompt).content
    return response

user_prompt = 'Explain Machine Learning?'
user_prompt = re.sub(r'[^0-9a-z-A-Z\s]',' ', user_prompt)
r_response = r_search(user_prompt)
g_response = g_text(r_response, user_prompt)
# print(g_response)

In [45]:
print(g_response[0]['text'])

**Input:** Explain Machine Learning 

**Output:** 

### Machine Learning Workflow: A Gradient Boosting Perspective

```
[Input Data] ──> [Initialize F₀(x)] ──> [Iterative Loop (m = 1 to M)] ──> [Final Ensemble F_M(x)]
                                               │             ▲
                                               ▼             │
                                         [Compute r_im] ──> [Train h_m(x)]
```

---

#### Step 1: Initialization — Defining the Baseline Model ($F_0(x)$)
*   **Concept:** Before learning complex patterns, we establish a baseline. In Machine Learning, we start with a simple guess (e.g., predicting the average outcome for all data points).
*   **Gradient Boosting Translation:** Initialize the model with a constant value that minimizes the loss function:
    $$F_0(x) = \arg\min_{\gamma} \sum_{i=1}^{N} L(y_i, \gamma)$$
    *Where $L$ is the loss function, $y_i$ is the actual target, and $\gamma$ is the initial prediction.*

---

#### Step 2: The Iterat

In [46]:
# print(response['choices'][0]['message']['content'])

# 31 JULY 2026

Another method

In [47]:
# generation part